# 03 — Granularity Correlations

Reproduces paper **Fig. 4**
(`all_models_gran_steerability_PV_scatter.png`,
`all_models_gran_T95_scatter.png`), **Fig. 7**
(`per-model-gran-steerabilitty.png`, sic), **Fig. 8**
(`per-model-gran-T95.png`).

Granularity is the paper's `pl_cross_within` formulation
G_c = mean_ℓ(γ_c(ℓ) / 𝒜_c(ℓ)), implemented in
`grace/diagnostics/granularity.py`.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from grace.analysis.load_results import load_summary_results
from grace.analysis.t95 import t95
from grace.diagnostics.granularity import granularity

FIG_DIR = Path('Images'); FIG_DIR.mkdir(exist_ok=True)
MODELS = {
    'gemma2': 'google/gemma-2-2b-it',
    'gemma3': 'google/gemma-3-27b-it',
    'llama3': 'meta-llama/Llama-3.3-70B-Instruct',
}
CONCEPTS = sorted(p.stem for p in Path('concepts/gpt-5/extract').glob('*.json'))

In [ ]:
rows = []
for tag, model_name in MODELS.items():
    for c in CONCEPTS:
        try:
            G, _ = granularity(model_name, c)
        except FileNotFoundError:
            continue
        sums = load_summary_results(model_name, c, method='pv')
        best_utility = max((r['mean_utility'] for r in sums if r.get('mean_utility') is not None), default=None)
        mean_t95, _ = t95(model_name, c, method='pv', mode='unconstrained')
        rows.append({'model': tag, 'concept': c, 'granularity': G,
                     'best_utility': best_utility, 'T95': mean_t95})
df = pd.DataFrame(rows).dropna()
df.head()

## Fig. 4a — Pooled granularity vs. best-found steerability


In [ ]:
plt.figure(figsize=(5, 5))
for tag in MODELS:
    sub = df[df.model == tag]
    plt.scatter(sub.granularity, sub.best_utility, label=tag, alpha=0.7)
rho, p = spearmanr(df.granularity, df.best_utility)
plt.title(f'Granularity vs. best utility  (Spearman ρ={rho:.2f}, p={p:.1e})')
plt.xlabel('Granularity G_c'); plt.ylabel('Best-found utility (PV)'); plt.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / 'all_models_gran_steerability_PV_scatter.png', dpi=150); plt.show()

## Fig. 4b — Pooled granularity vs. T_95


In [ ]:
df2 = df.dropna(subset=['T95'])
plt.figure(figsize=(5, 5))
for tag in MODELS:
    sub = df2[df2.model == tag]
    plt.scatter(sub.granularity, sub.T95, label=tag, alpha=0.7)
rho, p = spearmanr(df2.granularity, df2.T95)
plt.title(f'Granularity vs. T_95  (Spearman ρ={rho:.2f}, p={p:.1e})')
plt.xlabel('Granularity G_c'); plt.ylabel('T_95 (trials to 95% best)'); plt.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / 'all_models_gran_T95_scatter.png', dpi=150); plt.show()

## Fig. 7/8 — Per-model breakouts (3-panel side-by-side)


In [ ]:
for ycol, fname in [('best_utility', 'per-model-gran-steerabilitty.png'), ('T95', 'per-model-gran-T95.png')]:
    sub = df.dropna(subset=[ycol])
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
    for ax, tag in zip(axes, MODELS):
        s = sub[sub.model == tag]
        ax.scatter(s.granularity, s[ycol])
        if len(s) > 2:
            r, p = spearmanr(s.granularity, s[ycol])
            ax.set_title(f'{tag}\nρ={r:.2f}, p={p:.1e}')
        else:
            ax.set_title(tag)
        ax.set_xlabel('G_c')
    axes[0].set_ylabel(ycol)
    plt.tight_layout(); plt.savefig(FIG_DIR / fname, dpi=150); plt.show()